# DATATHON 2026 — Merged EDA + XAI Dashboards v4

Bản này gộp EDA + XAI Forecast đúng hơn cho đề:
- Giữ style biểu đồ gần notebook gốc (`sns.whitegrid`, palette `husl` / `Set2`, layout 2x2 / 3x2 / full-width).
- Thêm forecast đến đúng giai đoạn test 2023-01-01 → 2024-07-01.
- Dashboard 3 đã ghép `XAI-driven Forecast to 2024` vào hàng trên.
- XAI model dùng LightGBM + SHAP, có validation MAE/RMSE/R² và xuất `submission_xai_2024.csv`.
- Đã sửa leakage: không dùng `Revenue`, `COGS`, `gross_margin` cùng ngày làm feature dự báo; chỉ dùng calendar, lag/rolling history và driver dạng lag/profile từ dữ liệu train.

Chạy cell cuối để xuất 3 dashboard PNG và file submission vào `dashboard_outputs/`.


## 0. Imports

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import lightgbm as lgb
    import shap
    HAS_XAI = True
    XAI_IMPORT_ERROR = None
except Exception as e:
    lgb = None
    shap = None
    HAS_XAI = False
    XAI_IMPORT_ERROR = e

SEED = 42
np.random.seed(SEED)


## 0. CONFIG

In [2]:
# =========================
# 0. CONFIG
# =========================

# Sửa dòng này nếu CSV nằm ở thư mục khác.
# Ví dụ Windows: DATA_DIR = r"D:/datathon/raw"
DATA_DIR = os.getenv("DATA_DIR", "../mycode/raw")

OUT_DIR = Path("dashboard_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Style gốc từ 2 notebook EDA/XAI:
# - seaborn whitegrid
# - palette husl / Set2
# - màu chủ đạo giống notebook gốc
sns.set_theme(style="whitegrid", palette="husl")

BG = "#FFFFFF"
PANEL = "#FFFFFF"
BORDER = "#E5E7EB"
TEXT = "#111827"
SUB = "#6B7280"
GRID = "#D1D5DB"

A1 = "#2196F3"   # BLUE
A2 = "#FF5722"   # RED/ORANGE
A3 = "#4CAF50"   # GREEN
A4 = "#FF9800"   # ORANGE
A5 = "#9C27B0"   # PURPLE
A6 = "#00BCD4"   # CYAN
A7 = "#F44336"   # RED
A8 = "#607D8B"   # BLUE GREY
GRAY = "#6B7280"

BLUE   = A1
RED    = A2
GREEN  = A3
ORANGE = A4
PURPLE = A5

PALETTE = [A1, A2, A3, A4, A5, A6]
CAT_C = PALETTE + ["#3F51B5", "#009688", "#795548", "#E91E63"]

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
    "savefig.facecolor": BG,
    "figure.facecolor": BG,
    "axes.facecolor": PANEL,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.titleweight": "bold",
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## 1. HELPERS

In [3]:
# =========================
# 1. HELPERS
# =========================

def fmt_vnd(x, pos=None):
    if pd.isna(x):
        return ""
    x = float(x)
    ax = abs(x)
    if ax >= 1e9:
        return f"{x/1e9:.1f}B"
    if ax >= 1e6:
        return f"{x/1e6:.1f}M"
    if ax >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"


def fmt_pct(x, pos=None):
    if pd.isna(x):
        return ""
    return f"{x:.0%}"


def safe_div(a, b):
    return np.where(np.asarray(b) == 0, np.nan, np.asarray(a) / np.asarray(b))


def add_bar_labels(ax, values=None, fmt="{:.1f}", fontsize=8, pad=3, horizontal=False):
    if values is None:
        patches = ax.patches
        for p in patches:
            if horizontal:
                val = p.get_width()
                ax.text(val, p.get_y() + p.get_height()/2, fmt.format(val), va="center", ha="left", fontsize=fontsize, color=SUB)
            else:
                val = p.get_height()
                ax.text(p.get_x() + p.get_width()/2, val, fmt.format(val), va="bottom", ha="center", fontsize=fontsize, color=SUB)
    else:
        for p, val in zip(ax.patches, values):
            if horizontal:
                ax.text(p.get_width(), p.get_y() + p.get_height()/2, fmt.format(val), va="center", ha="left", fontsize=fontsize, color=SUB)
            else:
                ax.text(p.get_x() + p.get_width()/2, p.get_height(), fmt.format(val), va="bottom", ha="center", fontsize=fontsize, color=SUB)


def annotate_empty(ax, title, msg):
    ax.set_title(title, loc="left", fontweight="bold")
    ax.text(0.5, 0.5, msg, ha="center", va="center", wrap=True, transform=ax.transAxes, color=SUB, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def resolve_data_dir(data_dir=DATA_DIR):
    """Tìm thư mục chứa CSV. Không giả định path cứng."""
    required_probe = [
        "sales.csv", "orders.csv", "order_items.csv", "products.csv",
        "customers.csv", "payments.csv", "inventory.csv", "web_traffic.csv"
    ]

    candidates = []
    if data_dir:
        candidates.append(Path(data_dir))
    env_dir = os.getenv("DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))

    candidates.extend([
        Path.cwd(),
        Path.cwd() / "raw",
        Path.cwd() / "data",
        Path.cwd() / "input",
        Path.cwd().parent / "raw",
        Path.cwd().parent / "data",
        Path("/mnt/data"),
        Path("/mnt/data/raw"),
        Path("/mnt/data/data"),
        Path("../mycode/raw"),
    ])

    scored = []
    for p in candidates:
        try:
            count = sum((p / f).exists() for f in required_probe)
            scored.append((count, p))
        except Exception:
            pass

    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    if not scored or scored[0][0] == 0:
        raise FileNotFoundError(
            "Không tìm thấy các CSV của Datathon. Hãy đặt notebook/script cùng thư mục với CSV "
            "hoặc sửa biến DATA_DIR ở đầu file. Tối thiểu cần sales.csv, orders.csv, order_items.csv, products.csv."
        )
    return scored[0][1]


def read_csv_if_exists(data_dir, filename, parse_dates=None):
    path = Path(data_dir) / filename
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=parse_dates or [])


def load_data(data_dir=DATA_DIR):
    data_dir = resolve_data_dir(data_dir)
    print(f"Using DATA_DIR = {data_dir.resolve()}")

    inventory = read_csv_if_exists(data_dir, "inventory_enhanced.csv", ["snapshot_date"])
    if inventory is None:
        inventory = read_csv_if_exists(data_dir, "inventory.csv", ["snapshot_date"])

    data = {
        "products": read_csv_if_exists(data_dir, "products.csv"),
        "customers": read_csv_if_exists(data_dir, "customers.csv", ["signup_date"]),
        "promotions": read_csv_if_exists(data_dir, "promotions.csv", ["start_date", "end_date"]),
        "geography": read_csv_if_exists(data_dir, "geography.csv"),
        "orders": read_csv_if_exists(data_dir, "orders.csv", ["order_date"]),
        "order_items": read_csv_if_exists(data_dir, "order_items.csv"),
        "payments": read_csv_if_exists(data_dir, "payments.csv"),
        "shipments": read_csv_if_exists(data_dir, "shipments.csv", ["ship_date", "delivery_date"]),
        "returns": read_csv_if_exists(data_dir, "returns.csv", ["return_date"]),
        "reviews": read_csv_if_exists(data_dir, "reviews.csv", ["review_date"]),
        "sales": read_csv_if_exists(data_dir, "sales.csv", ["Date"]),
        "sample_submission": read_csv_if_exists(data_dir, "sample_submission.csv", ["Date"]),
        "sales_test": read_csv_if_exists(data_dir, "sales_test.csv", ["Date"]),
        "inventory": inventory,
        "web_traffic": read_csv_if_exists(data_dir, "web_traffic.csv", ["date"]),
    }

    must_have = ["sales", "orders", "order_items", "products", "customers"]
    missing = [k for k in must_have if data.get(k) is None]
    if missing:
        raise FileNotFoundError(f"Thiếu file bắt buộc: {missing}")

    return data


## 2. PREPARE BASE TABLES

In [4]:
def load_data(data_dir=DATA_DIR):
    data_dir = resolve_data_dir(data_dir)
    print(f"Using DATA_DIR = {data_dir.resolve()}")

    inventory = read_csv_if_exists(data_dir, "inventory_enhanced.csv", ["snapshot_date"])
    if inventory is None:
        inventory = read_csv_if_exists(data_dir, "inventory.csv", ["snapshot_date"])

    data = {
        "products": read_csv_if_exists(data_dir, "products.csv"),
        "customers": read_csv_if_exists(data_dir, "customers.csv", ["signup_date"]),
        "promotions": read_csv_if_exists(data_dir, "promotions.csv", ["start_date", "end_date"]),
        "geography": read_csv_if_exists(data_dir, "geography.csv"),
        "orders": read_csv_if_exists(data_dir, "orders.csv", ["order_date"]),
        "order_items": read_csv_if_exists(data_dir, "order_items.csv"),
        "payments": read_csv_if_exists(data_dir, "payments.csv"),
        "shipments": read_csv_if_exists(data_dir, "shipments.csv", ["ship_date", "delivery_date"]),
        "returns": read_csv_if_exists(data_dir, "returns.csv", ["return_date"]),
        "reviews": read_csv_if_exists(data_dir, "reviews.csv", ["review_date"]),
        "sales": read_csv_if_exists(data_dir, "sales.csv", ["Date"]),
        "sample_submission": read_csv_if_exists(data_dir, "sample_submission.csv", ["Date"]),
        "sales_test": read_csv_if_exists(data_dir, "sales_test.csv", ["Date"]),
        "inventory": inventory,
        "web_traffic": read_csv_if_exists(data_dir, "web_traffic.csv", ["date"]),
    }

    must_have = ["sales", "orders", "order_items", "products", "customers"]
    missing = [k for k in must_have if data.get(k) is None]
    if missing:
        raise FileNotFoundError(f"Thiếu file bắt buộc: {missing}")

    return data


# =========================
# 2. PREPARE BASE TABLES
# =========================

def prepare_base(data):
    products = data["products"].copy()
    customers = data["customers"].copy()
    orders = data["orders"].copy()
    order_items = data["order_items"].copy()
    payments = data.get("payments")
    returns = data.get("returns")
    inventory = data.get("inventory")
    web_traffic = data.get("web_traffic")
    sample_submission = data.get("sample_submission")
    sales_test = data.get("sales_test")
    sales = data["sales"].copy()

    # Sales
    sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")
    sales = sales.dropna(subset=["Date"]).sort_values("Date").copy()
    sales["year"] = sales["Date"].dt.year
    sales["month"] = sales["Date"].dt.month
    sales["ym"] = sales["Date"].dt.to_period("M").astype(str)
    sales["gross_profit"] = sales["Revenue"] - sales["COGS"]
    sales["gross_margin"] = safe_div(sales["gross_profit"], sales["Revenue"])

    # Orders
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
    orders["ym"] = orders["order_date"].dt.to_period("M").astype(str)
    orders["order_day"] = orders["order_date"].dt.normalize()

    # Order items + products
    oi_prod = order_items.merge(products, on="product_id", how="left")
    oi_prod["line_revenue"] = oi_prod["quantity"] * oi_prod["unit_price"]
    if "cogs" in oi_prod.columns:
        oi_prod["line_cogs"] = oi_prod["quantity"] * oi_prod["cogs"]
        oi_prod["line_gross_profit"] = oi_prod["line_revenue"] - oi_prod["line_cogs"]
    else:
        oi_prod["line_cogs"] = np.nan
        oi_prod["line_gross_profit"] = np.nan
    oi_prod["has_promo"] = oi_prod.get("promo_id", pd.Series(index=oi_prod.index, dtype=object)).notna()
    if "promo_id_2" in oi_prod.columns:
        oi_prod["has_promo"] = oi_prod["has_promo"] | oi_prod["promo_id_2"].notna()

    oi_ord = oi_prod.merge(
        orders[["order_id", "order_date", "order_day", "ym", "customer_id", "order_source", "order_status"]],
        on="order_id",
        how="left",
        suffixes=("", "_order")
    )

    # Category revenue
    cat_rev = None
    if "category" in oi_ord.columns:
        cat_rev = oi_ord.groupby("category", dropna=False)["line_revenue"].sum().sort_values(ascending=True)

    # Inventory monthly
    inv_month = None
    if inventory is not None:
        inventory = inventory.copy()
        if "snapshot_date" in inventory.columns:
            inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"], errors="coerce")
            inventory["ym"] = inventory["snapshot_date"].dt.to_period("M").astype(str)
            inventory["year"] = inventory["snapshot_date"].dt.year
            inventory["month"] = inventory["snapshot_date"].dt.month
        elif {"year", "month"}.issubset(inventory.columns):
            inventory["ym"] = pd.PeriodIndex(year=inventory["year"], month=inventory["month"], freq="M").astype(str)

        agg = {}
        if "stockout_flag" in inventory.columns:
            agg["stockout_rate"] = ("stockout_flag", "mean")
        if "stockout_days" in inventory.columns:
            agg["stockout_days"] = ("stockout_days", "sum")
        if "fill_rate" in inventory.columns:
            agg["fill_rate"] = ("fill_rate", "mean")
        if "sell_through_rate" in inventory.columns:
            agg["sell_through_rate"] = ("sell_through_rate", "mean")
        if agg:
            inv_month = inventory.groupby("ym").agg(**agg).reset_index()

    # Monthly sales
    sales_month = sales.groupby("ym").agg(
        Revenue=("Revenue", "sum"),
        COGS=("COGS", "sum"),
        gross_profit=("gross_profit", "sum"),
    ).reset_index()
    sales_month["gross_margin"] = safe_div(sales_month["gross_profit"], sales_month["Revenue"])
    sales_month["date"] = pd.to_datetime(sales_month["ym"] + "-01")
    if inv_month is not None:
        sales_inv_month = sales_month.merge(inv_month, on="ym", how="left")
    else:
        sales_inv_month = sales_month.copy()

    # Promo day
    promo_day = (
        oi_ord.groupby("order_day")
        .agg(
            promo_lines=("has_promo", "sum"),
            total_lines=("order_id", "size"),
            total_discount=("discount_amount", "sum") if "discount_amount" in oi_ord.columns else ("has_promo", "sum"),
            max_discount=("discount_amount", "max") if "discount_amount" in oi_ord.columns else ("has_promo", "max"),
        )
        .reset_index()
        .rename(columns={"order_day": "Date"})
    )
    promo_day["has_promo_day"] = promo_day["promo_lines"] > 0
    sales_promo = sales.merge(promo_day[["Date", "has_promo_day", "total_discount", "max_discount"]], on="Date", how="left")
    sales_promo["has_promo_day"] = sales_promo["has_promo_day"].fillna(False)
    sales_promo["total_discount"] = sales_promo["total_discount"].fillna(0)
    sales_promo["max_discount"] = sales_promo["max_discount"].fillna(0)

    # Cohort buyer retention đúng nghĩa
    # Base = khách có mua ở M+0
    # Cancelled không tính là buyer

    customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
    orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

    if "order_status" in orders.columns:
        orders_valid = orders[orders["order_status"].ne("cancelled")].copy()
    else:
        orders_valid = orders.copy()

    customers_cohort = customers[["customer_id", "signup_date"]].copy()
    customers_cohort = customers_cohort.dropna(subset=["signup_date"])
    customers_cohort["cohort_month"] = customers_cohort["signup_date"].dt.to_period("M")

    orders_cust = orders_valid.merge(
        customers_cohort[["customer_id", "cohort_month"]],
        on="customer_id",
        how="inner"
    )

    orders_cust = orders_cust.dropna(subset=["order_date"]).copy()
    orders_cust["order_period"] = orders_cust["order_date"].dt.to_period("M")

    orders_cust["period_number"] = (
        orders_cust["order_period"].apply(lambda x: x.ordinal)
        - orders_cust["cohort_month"].apply(lambda x: x.ordinal)
    )

    orders_cust = orders_cust[orders_cust["period_number"] >= 0].copy()

    m0_buyers = (
        orders_cust[orders_cust["period_number"] == 0]
        [["cohort_month", "customer_id"]]
        .drop_duplicates()
    )

    retained_orders = orders_cust.merge(
        m0_buyers,
        on=["cohort_month", "customer_id"],
        how="inner"
    )

    cohort_data = (
        retained_orders
        .groupby(["cohort_month", "period_number"])["customer_id"]
        .nunique()
        .reset_index()
    )

    cohort_pivot = cohort_data.pivot_table(
        index="cohort_month",
        columns="period_number",
        values="customer_id"
    )

    m0_size = (
        m0_buyers
        .groupby("cohort_month")["customer_id"]
        .nunique()
    )

    buyer_retention = cohort_pivot.divide(m0_size, axis=0) * 100
    buyer_retention_12 = buyer_retention.reindex(columns=range(12)).sort_index()

    max_order_period = orders_cust["order_period"].max()

    for cohort in buyer_retention_12.index:
        max_available = max_order_period.ordinal - cohort.ordinal
        max_available = min(max_available, 11)

        if max_available >= 0:
            observed_cols = list(range(max_available + 1))
            buyer_retention_12.loc[cohort, observed_cols] = (
                buyer_retention_12.loc[cohort, observed_cols].fillna(0)
            )

        if max_available < 11:
            future_cols = list(range(max_available + 1, 12))
            buyer_retention_12.loc[cohort, future_cols] = np.nan
    

    # Inter-order gap dùng luôn buyer_orders này
    ord_sorted = orders_cust.sort_values(["customer_id", "order_date"]).copy()
    ord_sorted["prev_order_date"] = ord_sorted.groupby("customer_id")["order_date"].shift(1)
    ord_sorted["gap_days"] = (ord_sorted["order_date"] - ord_sorted["prev_order_date"]).dt.days
    repeat_gaps = ord_sorted["gap_days"].dropna()
    repeat_gaps = repeat_gaps[repeat_gaps >= 0]
    
    repeat_gaps = ord_sorted["gap_days"].dropna()
    repeat_gaps = repeat_gaps[repeat_gaps >= 0]

    # RFM
    if payments is not None and "payment_value" in payments.columns:
        order_value = orders[["order_id", "customer_id", "order_date"]].merge(
            payments[["order_id", "payment_value"]], on="order_id", how="left"
        )
    else:
        value_by_order = oi_ord.groupby("order_id")["line_revenue"].sum().rename("payment_value").reset_index()
        order_value = orders[["order_id", "customer_id", "order_date"]].merge(value_by_order, on="order_id", how="left")

    snapshot_date = orders["order_date"].max() + pd.Timedelta(days=1)
    rfm = order_value.groupby("customer_id").agg(
        recency=("order_date", lambda s: (snapshot_date - s.max()).days),
        frequency=("order_id", "nunique"),
        monetary=("payment_value", "sum"),
    ).reset_index()
    all_customers = customers[["customer_id"]].drop_duplicates()
    rfm = all_customers.merge(rfm, on="customer_id", how="left")
    rfm["frequency"] = rfm["frequency"].fillna(0)
    rfm["monetary"] = rfm["monetary"].fillna(0)
    rfm["recency"] = rfm["recency"].fillna((snapshot_date - customers["signup_date"].min()).days if "signup_date" in customers else rfm["recency"].max())

    def qscore(s, reverse=False, bins=5):
        ranked = s.rank(method="first")
        labels = list(range(1, bins + 1))
        if reverse:
            labels = labels[::-1]
        try:
            return pd.qcut(ranked, bins, labels=labels).astype(int)
        except Exception:
            return pd.Series(np.ceil(ranked / max(len(s), 1) * bins).clip(1, bins), index=s.index).astype(int)

    purchased_mask = rfm["frequency"] > 0
    rfm["R"] = 0
    rfm["F"] = 0
    rfm["M"] = 0
    if purchased_mask.sum() > 0:
        rfm.loc[purchased_mask, "R"] = qscore(rfm.loc[purchased_mask, "recency"], reverse=True)
        rfm.loc[purchased_mask, "F"] = qscore(rfm.loc[purchased_mask, "frequency"], reverse=False)
        rfm.loc[purchased_mask, "M"] = qscore(rfm.loc[purchased_mask, "monetary"], reverse=False)

    def segment_row(row):
        if row["frequency"] <= 0:
            return "Never Purchased"
        r, f, m = row["R"], row["F"], row["M"]
        if r >= 4 and f >= 4 and m >= 4:
            return "Champions"
        if r >= 3 and f >= 3:
            return "Loyal"
        if r >= 4 and f <= 2:
            return "New / Potential"
        if r <= 2 and f >= 3:
            return "At Risk"
        if r <= 2 and f <= 2:
            return "Lost"
        return "Potential"

    rfm["rfm_segment"] = rfm.apply(segment_row, axis=1)
    retention = buyer_retention_12
    
    return {
        "sales": sales,
        "products": products,
        "customers": customers,
        "orders": orders,
        "order_items": order_items,
        "oi_ord": oi_ord,
        "cat_rev": cat_rev,
        "sales_month": sales_month,
        "sales_inv_month": sales_inv_month,
        "sales_promo": sales_promo,
        "retention": retention,
        "repeat_gaps": repeat_gaps,
        "rfm": rfm,
        "web_traffic": web_traffic,
        "inventory": inventory,
        "sample_submission": sample_submission,
        "sales_test": sales_test,
    }


## 3. XAI MODEL + SHAP

In [5]:
def feature_group(name):
    n = name.lower()
    if "n_orders" in n or "n_lines" in n:
        return "Order momentum"
    if "discount" in n or "promo" in n:
        return "Promotion"
    if "session" in n or "bounce" in n or "traffic" in n or "visitor" in n or "conversion" in n:
        return "Web traffic"
    if "stockout" in n or "fill_rate" in n or "sell_through" in n or "supply" in n:
        return "Inventory"
    if n.startswith("lag_") or n.startswith("rmean") or n.startswith("rstd") or n.startswith("ewm"):
        return "Lag / rolling history"
    if "cogs_lag" in n or "cogs_rmean" in n or "cogs_ewm" in n:
        return "Cost history"
    if n in {"year", "month", "day", "dayofweek", "dayofyear", "quarter", "weekofyear", "is_weekend", "is_month_start", "is_month_end"} or "sin" in n or "cos" in n:
        return "Calendar / seasonality"
    return "Other"


def _make_calendar_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["Date"], errors="coerce")
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["day"] = dt.dt.day
    df["dayofweek"] = dt.dt.dayofweek
    df["dayofyear"] = dt.dt.dayofyear
    df["quarter"] = dt.dt.quarter
    try:
        df["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    except Exception:
        df["weekofyear"] = 0
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = dt.dt.is_month_start.astype(int)
    df["is_month_end"] = dt.dt.is_month_end.astype(int)

    df["sin_yr"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["cos_yr"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)
    df["sin_mo"] = np.sin(2 * np.pi * df["day"] / 30.5)
    df["cos_mo"] = np.cos(2 * np.pi * df["day"] / 30.5)
    df["sin_wk"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["cos_wk"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    return df


def _build_daily_driver_table(base):
    """Tạo bảng driver theo ngày từ các bảng nội bộ, sau đó chỉ dùng dạng lag/profile khi forecast.
    Không dùng Revenue/COGS tương lai và không dùng dữ liệu ngoài.
    """
    sales = base["sales"][["Date"]].copy()
    min_date, max_date = sales["Date"].min(), sales["Date"].max()
    full_dates = pd.DataFrame({"Date": pd.date_range(min_date, max_date, freq="D")})
    driver = full_dates.copy()

    # Daily orders / promotions
    oi_ord = base.get("oi_ord")
    if oi_ord is not None and len(oi_ord) > 0:
        tmp = oi_ord.copy()
        if "order_day" not in tmp.columns:
            tmp["order_day"] = pd.to_datetime(tmp["order_date"], errors="coerce").dt.normalize()
        if "discount_amount" not in tmp.columns:
            tmp["discount_amount"] = 0.0
        if "has_promo" not in tmp.columns:
            tmp["has_promo"] = tmp.get("promo_id", pd.Series(index=tmp.index)).notna()
            if "promo_id_2" in tmp.columns:
                tmp["has_promo"] = tmp["has_promo"] | tmp["promo_id_2"].notna()
        daily_orders = (
            tmp.groupby("order_day")
            .agg(
                n_orders=("order_id", "nunique"),
                n_lines=("order_id", "size"),
                n_promos=("has_promo", "sum"),
                total_discount=("discount_amount", "sum"),
                max_discount=("discount_amount", "max"),
                avg_discount=("discount_amount", "mean"),
            )
            .reset_index()
            .rename(columns={"order_day": "Date"})
        )
        driver = driver.merge(daily_orders, on="Date", how="left")

    # Web traffic daily
    wt = base.get("web_traffic")
    if wt is not None and len(wt) > 0:
        wt = wt.copy()
        wt["date"] = pd.to_datetime(wt["date"], errors="coerce").dt.normalize()
        wt_agg = {}
        if "sessions" in wt.columns:
            wt_agg["total_sessions"] = ("sessions", "sum")
        if "unique_visitors" in wt.columns:
            wt_agg["unique_visitors"] = ("unique_visitors", "sum")
        if "page_views" in wt.columns:
            wt_agg["page_views"] = ("page_views", "sum")
        if "bounce_rate" in wt.columns:
            wt_agg["avg_bounce"] = ("bounce_rate", "mean")
        if "conversion_rate" in wt.columns:
            wt_agg["avg_conversion_rate"] = ("conversion_rate", "mean")
        if wt_agg:
            wt_daily = wt.groupby("date").agg(**wt_agg).reset_index().rename(columns={"date": "Date"})
            driver = driver.merge(wt_daily, on="Date", how="left")

    # Inventory monthly mapped to daily by month
    sim = base.get("sales_inv_month")
    if sim is not None and len(sim) > 0:
        inv_cols = [c for c in ["stockout_rate", "stockout_days", "fill_rate", "sell_through_rate"] if c in sim.columns]
        if inv_cols:
            inv_map = sim[["ym"] + inv_cols].copy()
            driver["ym"] = driver["Date"].dt.to_period("M").astype(str)
            driver = driver.merge(inv_map, on="ym", how="left")
            driver = driver.drop(columns=["ym"], errors="ignore")

    # Fill observed driver missing safely
    for c in driver.columns:
        if c != "Date" and pd.api.types.is_numeric_dtype(driver[c]):
            if c in {"n_orders", "n_lines", "n_promos", "total_discount", "max_discount", "avg_discount",
                     "total_sessions", "unique_visitors", "page_views"}:
                driver[c] = driver[c].fillna(0)
            else:
                driver[c] = driver[c].ffill().bfill().fillna(0)

    return driver


def _future_dates_from_submission(base):
    """Ưu tiên Date trong sample_submission.csv để giữ đúng thứ tự nộp bài.
    Nếu không có sample_submission, sinh đúng cửa sổ test theo đề: 2023-01-01 -> 2024-07-01.
    """
    for key in ["sample_submission", "sales_test"]:
        df = base.get(key)
        if df is not None and "Date" in df.columns:
            out = df[["Date"]].copy()
            out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
            out = out.dropna(subset=["Date"]).copy()
            if len(out) > 0:
                return out.reset_index(drop=True)

    start = pd.Timestamp("2023-01-01")
    end = pd.Timestamp("2024-07-01")
    last_train = pd.to_datetime(base["sales"]["Date"]).max()
    if last_train >= start:
        start = last_train + pd.Timedelta(days=1)
    return pd.DataFrame({"Date": pd.date_range(start, end, freq="D")})


def _extend_drivers_to_future(driver_hist, future_dates):
    """Ước lượng driver tương lai bằng seasonal profile từ lịch sử.
    Đây không phải dùng dữ liệu ngoài: chỉ lấy median theo tháng-ngày / tháng-thứ / tháng từ train.
    """
    driver_hist = driver_hist.copy()
    future = future_dates[["Date"]].copy()
    future["Date"] = pd.to_datetime(future["Date"], errors="coerce")

    hist = driver_hist.copy()
    hist["month"] = hist["Date"].dt.month
    hist["day"] = hist["Date"].dt.day
    hist["dayofweek"] = hist["Date"].dt.dayofweek

    future["month"] = future["Date"].dt.month
    future["day"] = future["Date"].dt.day
    future["dayofweek"] = future["Date"].dt.dayofweek

    driver_cols = [c for c in driver_hist.columns if c != "Date"]
    for col in driver_cols:
        md_map = hist.groupby(["month", "day"])[col].median()
        mdow_map = hist.groupby(["month", "dayofweek"])[col].median()
        m_map = hist.groupby("month")[col].median()
        global_median = hist[col].median()

        vals = []
        for _, r in future.iterrows():
            key_md = (r["month"], r["day"])
            key_mdow = (r["month"], r["dayofweek"])
            if key_md in md_map.index and pd.notna(md_map.loc[key_md]):
                vals.append(md_map.loc[key_md])
            elif key_mdow in mdow_map.index and pd.notna(mdow_map.loc[key_mdow]):
                vals.append(mdow_map.loc[key_mdow])
            elif r["month"] in m_map.index and pd.notna(m_map.loc[r["month"]]):
                vals.append(m_map.loc[r["month"]])
            else:
                vals.append(global_median if pd.notna(global_median) else 0)
        future[col] = vals

    future = future.drop(columns=["month", "day", "dayofweek"], errors="ignore")
    return pd.concat([driver_hist, future], ignore_index=True).sort_values("Date")


def _add_forecast_safe_features(df, driver_cols):
    """Feature set forecast-safe:
    - Không dùng Revenue/COGS cùng ngày.
    - Chỉ dùng calendar, lag/rolling history, và driver dạng lag.
    """
    df = df.copy().sort_values("Date").reset_index(drop=True)
    df = _make_calendar_features(df)
    df["log_revenue"] = np.log1p(df["Revenue"])
    df["log_cogs"] = np.log1p(df["COGS"])

    for lag in [1, 2, 3, 7, 14, 30, 60, 90, 180, 365]:
        df[f"lag_{lag}"] = df["log_revenue"].shift(lag)
        df[f"cogs_lag_{lag}"] = df["log_cogs"].shift(lag)

    for win in [7, 14, 30, 90]:
        df[f"rmean_{win}"] = df["log_revenue"].shift(1).rolling(win, min_periods=max(2, win // 3)).mean()
        df[f"cogs_rmean_{win}"] = df["log_cogs"].shift(1).rolling(win, min_periods=max(2, win // 3)).mean()

    df["rstd_30"] = df["log_revenue"].shift(1).rolling(30, min_periods=10).std()
    df["ewm_30"] = df["log_revenue"].shift(1).ewm(span=30, adjust=False).mean()
    df["cogs_ewm_30"] = df["log_cogs"].shift(1).ewm(span=30, adjust=False).mean()

    # Driver chỉ dùng ở dạng lag để forecast không cần biết cùng-ngày tương lai.
    for col in driver_cols:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        df[f"{col}_lag1"] = df[col].shift(1)
        df[f"{col}_lag7"] = df[col].shift(7)
        if col in {"n_orders", "n_lines", "n_promos", "total_discount", "max_discount",
                   "total_sessions", "unique_visitors", "page_views", "stockout_days"}:
            df[f"{col}_rmean7"] = df[col].shift(1).rolling(7, min_periods=2).mean()
            df[f"{col}_rmean30"] = df[col].shift(1).rolling(30, min_periods=10).mean()

    return df


def build_model_frame(base):
    sales = base["sales"][["Date", "Revenue", "COGS"]].copy().sort_values("Date")
    driver = _build_daily_driver_table(base)
    driver_cols = [c for c in driver.columns if c != "Date"]

    df = sales.merge(driver, on="Date", how="left")
    for c in driver_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].ffill().bfill().fillna(0)

    df = _add_forecast_safe_features(df, driver_cols)

    # Cấm leakage: exclude target, same-day driver raw, và các biến dùng trực tiếp doanh thu/COGS cùng ngày.
    exclude = {"Date", "Revenue", "COGS", "log_revenue", "log_cogs"}
    exclude.update(driver_cols)
    feature_cols = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

    model_df = df.dropna(subset=["log_revenue", "log_cogs"]).copy()
    return model_df, feature_cols, driver, driver_cols


def _fit_lgb_regressor(X_train, y_train, target_name="Revenue"):
    params = dict(
        objective="regression",
        n_estimators=1200,
        learning_rate=0.025,
        num_leaves=31,
        min_child_samples=25,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=1.2,
        random_state=SEED,
        verbosity=-1,
    )
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    return model


def _recursive_forecast_to_2024(base, model_rev, model_cogs, feature_cols, feature_medians, driver_hist, driver_cols, rev_bias=0.0, cogs_bias=0.0, band_pct=0.12):
    future_dates = _future_dates_from_submission(base)
    if future_dates.empty:
        return pd.DataFrame(columns=["Date", "Revenue", "COGS"])

    driver_ext = _extend_drivers_to_future(driver_hist, future_dates)

    hist = base["sales"][["Date", "Revenue", "COGS"]].copy().sort_values("Date")
    future_blank = future_dates.copy()
    future_blank["Revenue"] = np.nan
    future_blank["COGS"] = np.nan

    ext = pd.concat([hist, future_blank], ignore_index=True).sort_values("Date").reset_index(drop=True)
    ext = ext.merge(driver_ext, on="Date", how="left")
    for c in driver_cols:
        if c in ext.columns and pd.api.types.is_numeric_dtype(ext[c]):
            ext[c] = ext[c].ffill().bfill().fillna(0)

    future_set = set(pd.to_datetime(future_dates["Date"]).dt.normalize())
    future_idx = ext.index[ext["Date"].dt.normalize().isin(future_set)].tolist()

    for idx in future_idx:
        feat_df = _add_forecast_safe_features(ext, driver_cols)
        x_row = feat_df.loc[[idx], feature_cols].replace([np.inf, -np.inf], np.nan)
        x_row = x_row.fillna(feature_medians).fillna(0)

        pred_rev = np.expm1(model_rev.predict(x_row)[0]) + rev_bias
        pred_cogs = np.expm1(model_cogs.predict(x_row)[0]) + cogs_bias

        pred_rev = max(float(pred_rev), 0.0)
        pred_cogs = max(float(pred_cogs), 0.0)

        # Giữ COGS hợp lý, tránh dự báo COGS âm hoặc vượt Revenue quá vô lý.
        if pred_rev > 0:
            pred_cogs = min(pred_cogs, pred_rev * 1.20)

        ext.loc[idx, "Revenue"] = pred_rev
        ext.loc[idx, "COGS"] = pred_cogs

    out = ext.loc[future_idx, ["Date", "Revenue", "COGS"]].copy()
    out["gross_profit"] = out["Revenue"] - out["COGS"]
    out["lower_12pct"] = np.maximum(out["Revenue"] * (1 - band_pct), 0)
    out["upper_12pct"] = out["Revenue"] * (1 + band_pct)
    out["forecast_month"] = out["Date"].dt.to_period("M").astype(str)
    return out.reset_index(drop=True)


def train_xai_model(base):
    if not HAS_XAI:
        print(f"Không import được LightGBM/SHAP: {XAI_IMPORT_ERROR}")
        return None

    model_df, feature_cols, driver_hist, driver_cols = build_model_frame(base)
    if len(model_df) < 500:
        print("Không đủ dòng sau khi tạo lag feature để train XAI model.")
        return None

    X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_rev = model_df["log_revenue"]
    y_cogs = model_df["log_cogs"]

    split = int(len(model_df) * 0.8)
    X_train_raw, X_valid_raw = X.iloc[:split], X.iloc[split:]
    y_rev_train, y_rev_valid = y_rev.iloc[:split], y_rev.iloc[split:]
    y_cogs_train, y_cogs_valid = y_cogs.iloc[:split], y_cogs.iloc[split:]

    feature_medians = X_train_raw.median(numeric_only=True).fillna(0)
    X_train = X_train_raw.fillna(feature_medians).fillna(0)
    X_valid = X_valid_raw.fillna(feature_medians).fillna(0)

    model_rev = _fit_lgb_regressor(X_train, y_rev_train, "Revenue")
    model_cogs = _fit_lgb_regressor(X_train, y_cogs_train, "COGS")

    pred_valid_log = model_rev.predict(X_valid)
    pred_valid = np.expm1(pred_valid_log)
    actual_valid = np.expm1(y_rev_valid)

    pred_cogs_valid_log = model_cogs.predict(X_valid)
    pred_cogs_valid = np.expm1(pred_cogs_valid_log)
    actual_cogs_valid = np.expm1(y_cogs_valid)

    metrics = {
        "mae": mean_absolute_error(actual_valid, pred_valid),
        "rmse": float(np.sqrt(mean_squared_error(actual_valid, pred_valid))),
        "r2": r2_score(actual_valid, pred_valid),
        "bias": float(np.mean(actual_valid - pred_valid)),
        "cogs_mae": mean_absolute_error(actual_cogs_valid, pred_cogs_valid),
        "cogs_bias": float(np.mean(actual_cogs_valid - pred_cogs_valid)),
    }

    # Dải rủi ro dùng residual validation; nếu quá hẹp/rộng thì chặn 8%-25%.
    rel_mae = metrics["mae"] / max(float(np.nanmean(actual_valid)), 1.0)
    band_pct = float(np.clip(rel_mae, 0.08, 0.25))

    n_samp = min(3000, len(X_train))
    X_samp = X_train.sample(n_samp, random_state=SEED)
    explainer = shap.TreeExplainer(model_rev)
    shap_values = explainer.shap_values(X_samp)
    shap_values = np.asarray(shap_values)

    shap_df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    shap_df["group"] = shap_df["feature"].map(feature_group)

    group_shap = (
        shap_df.groupby("group")["mean_abs_shap"]
        .sum()
        .sort_values(ascending=True)
        .reset_index()
    )
    group_shap["share"] = group_shap["mean_abs_shap"] / group_shap["mean_abs_shap"].sum()

    valid_df = model_df.iloc[split:][["Date", "Revenue", "COGS"]].copy()
    valid_df["pred"] = pred_valid
    valid_df["pred_optimized"] = np.maximum(pred_valid + metrics["bias"], 0)
    valid_df["pred_cogs"] = pred_cogs_valid
    valid_df["pred_cogs_optimized"] = np.maximum(pred_cogs_valid + metrics["cogs_bias"], 0)
    valid_df["error"] = valid_df["Revenue"] - valid_df["pred"]

    future_df = _recursive_forecast_to_2024(
        base=base,
        model_rev=model_rev,
        model_cogs=model_cogs,
        feature_cols=feature_cols,
        feature_medians=feature_medians,
        driver_hist=driver_hist,
        driver_cols=driver_cols,
        rev_bias=metrics["bias"],
        cogs_bias=metrics["cogs_bias"],
        band_pct=band_pct,
    )

    return {
        "model": model_rev,
        "model_cogs": model_cogs,
        "model_df": model_df,
        "feature_cols": feature_cols,
        "feature_medians": feature_medians,
        "driver_hist": driver_hist,
        "driver_cols": driver_cols,
        "X_samp": X_samp,
        "shap_values": shap_values,
        "shap_df": shap_df,
        "group_shap": group_shap,
        "valid_df": valid_df,
        "future_df": future_df,
        "metrics": metrics,
        "band_pct": band_pct,
    }


def save_xai_submission(xai, out_dir=OUT_DIR, filename="submission_xai_2024.csv"):
    """Xuất submission đúng format Date, Revenue, COGS theo sample_submission/test window."""
    if xai is None or "future_df" not in xai or xai["future_df"] is None or xai["future_df"].empty:
        return None
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    sub = xai["future_df"][["Date", "Revenue", "COGS"]].copy()
    sub["Date"] = pd.to_datetime(sub["Date"]).dt.strftime("%Y-%m-%d")
    sub["Revenue"] = sub["Revenue"].clip(lower=0)
    sub["COGS"] = sub["COGS"].clip(lower=0)
    path = out_dir / filename
    sub.to_csv(path, index=False)
    print(f"Saved submission: {path}")
    return path


# =========================
# 4. DASHBOARD 1
# =========================


## 4. DASHBOARD 1

In [6]:
def save_dashboard1(base, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    sales = base["sales"].copy()
    sales_inv = base["sales_inv_month"].copy()
    cat_rev = base["cat_rev"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    # 1A. Doanh thu theo năm — bar giống notebook gốc: peak màu nóng, còn lại xanh
    annual = sales.groupby("year")["Revenue"].sum().reset_index()
    colors_y = [RED if v == annual["Revenue"].max() else BLUE for v in annual["Revenue"]]
    bars = axes[0].bar(
        annual["year"], annual["Revenue"] / 1e9,
        color=colors_y, edgecolor="white", linewidth=0.8, alpha=0.85
    )
    axes[0].set_title("Doanh thu theo năm", fontweight="bold")
    axes[0].set_xlabel("Năm")
    axes[0].set_ylabel("Tổng doanh thu (Tỷ VND)")
    axes[0].tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, annual["Revenue"]):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val/1e9:.1f}B", ha="center", va="bottom", fontsize=8
        )

    # 1B. Doanh thu trung bình theo tháng — bar giống fig02 seasonality
    MONTH_NAMES = ["T1","T2","T3","T4","T5","T6","T7","T8","T9","T10","T11","T12"]
    monthly_avg = sales.groupby("month")["Revenue"].mean().reindex(range(1, 13))
    colors_m = [RED if v == monthly_avg.max() else BLUE for v in monthly_avg]
    bars = axes[1].bar(
        MONTH_NAMES, monthly_avg.values / 1e3,
        color=colors_m, edgecolor="white", linewidth=0.8, alpha=0.9
    )
    axes[1].set_title("Doanh thu trung bình theo tháng", fontweight="bold")
    axes[1].set_xlabel("Tháng")
    axes[1].set_ylabel("Doanh thu TB/ngày (Nghìn VND)")
    for i, v in enumerate(monthly_avg.values):
        if pd.notna(v):
            axes[1].text(i, v/1e3 + 0.5, f"{v/1e3:.0f}k", ha="center", va="bottom", fontsize=8)

    # 1C. Stock rate vs doanh thu tháng — bar + twin plot giống fig12_inventory
    ax = axes[2]
    if "stockout_rate" in sales_inv.columns and sales_inv["stockout_rate"].notna().any():
        ax_twin = ax.twinx()
        ax.bar(
            sales_inv["date"], sales_inv["stockout_rate"] * 100,
            width=20, alpha=0.6, color=RED, label="Stockout Rate (%)"
        )
        ax_twin.plot(
            sales_inv["date"], sales_inv["Revenue"] / 1e6,
            color=BLUE, linewidth=2, label="Doanh thu (M VND)"
        )
        ax.set_ylabel("Stockout Rate (%)", color=RED)
        ax_twin.set_ylabel("Doanh thu (Triệu VND)", color=BLUE)
        ax.set_title("Stockout Rate vs Doanh thu theo Tháng", fontweight="bold")
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax_twin.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    elif "stockout_days" in sales_inv.columns and sales_inv["stockout_days"].notna().any():
        ax_twin = ax.twinx()
        ax.bar(
            sales_inv["date"], sales_inv["stockout_days"],
            width=20, alpha=0.6, color=RED, label="Stockout Days"
        )
        ax_twin.plot(
            sales_inv["date"], sales_inv["Revenue"] / 1e6,
            color=BLUE, linewidth=2, label="Doanh thu (M VND)"
        )
        ax.set_ylabel("Stockout days", color=RED)
        ax_twin.set_ylabel("Doanh thu (Triệu VND)", color=BLUE)
        ax.set_title("Stockout Days vs Doanh thu theo Tháng", fontweight="bold")
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax_twin.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    else:
        annotate_empty(ax, "Stock rate vs doanh thu theo tháng", "Không có stockout_rate/stockout_days trong inventory.")

    # 1D. Doanh thu theo danh mục — horizontal bar giống fig03_product_analysis
    ax = axes[3]
    if cat_rev is not None and len(cat_rev) > 0:
        plot_cat = cat_rev.sort_values(ascending=False)
        colors_c = sns.color_palette("husl", len(plot_cat))
        bars = ax.barh(
            plot_cat.index.astype(str), plot_cat.values / 1e6,
            color=colors_c, edgecolor="white"
        )
        ax.set_xlabel("Tổng Doanh thu (Triệu VND)")
        ax.set_title("Doanh thu theo Danh mục Sản phẩm", fontweight="bold")
        ax.invert_yaxis()
        for bar, val in zip(bars, plot_cat.values):
            ax.text(
                bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{val/1e6:.1f}M", va="center", fontsize=9
            )
    else:
        annotate_empty(ax, "Doanh thu theo danh mục sản phẩm", "Không tạo được do thiếu category/order_items/products.")

    plt.suptitle("DASHBOARD 1 — Revenue, Seasonality, Inventory & Category", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = Path(out_dir) / "dashboard_1_revenue_inventory_category.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")
    return path


## 5. DASHBOARD 2

In [7]:
def save_dashboard2(base, xai=None, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    retention = base.get("retention")
    gaps = base.get("repeat_gaps")
    sales_month = base.get("sales_month")
    sales_promo = base.get("sales_promo")

    fig, axes = plt.subplots(3, 2, figsize=(20, 14))
    axes = axes.flatten()

    # 2A. SHAP importance theo nhóm feature
    ax = axes[0]
    if xai is not None and "group_shap" in xai and xai["group_shap"] is not None and len(xai["group_shap"]) > 0:
        g = xai["group_shap"].copy()
        g = g.sort_values("mean_abs_shap", ascending=False)
        palette = sns.color_palette("Set2", len(g))
        bars = ax.barh(g["group"][::-1], g["share"][::-1] * 100, color=palette[::-1], alpha=0.85)
        ax.set_xlabel("Tỷ trọng Mean |SHAP| (%)")
        ax.set_title("SHAP Importance theo Nhóm Feature", fontweight="bold")
        for bar, p in zip(bars, (g["share"][::-1] * 100)):
            ax.text(
                bar.get_width() * 1.01,
                bar.get_y() + bar.get_height() / 2,
                f"{p:.1f}%",
                va="center",
                fontsize=9,
            )
    else:
        annotate_empty(ax, "SHAP Importance theo Nhóm Feature", "Chưa có XAI model. Chạy train_xai_model(base).")

    # 2B. Buyer cohort retention heatmap
    ax = axes[1]
    if retention is not None and not retention.empty:
        ret_plot = retention.tail(24).copy()
        ret_plot = ret_plot.replace([np.inf, -np.inf], np.nan).clip(lower=0, upper=100)
        ret_plot.index = ret_plot.index.astype(str)
        ret_plot.columns = [f"M+{int(c)}" if isinstance(c, (int, np.integer)) else str(c) for c in ret_plot.columns]

        sns.heatmap(
            ret_plot,
            ax=ax,
            cmap="RdYlGn",
            fmt=".0f",
            annot=True,
            linewidths=0.3,
            vmin=0,
            vmax=100,
            cbar_kws={"label": "Buyer Retention Rate (%)"},
            annot_kws={"size": 7},
        )
        ax.set_title("Buyer Cohort Retention Rate", fontweight="bold")
        ax.set_xlabel("Tháng sau cohort")
        ax.set_ylabel("Cohort month")
    else:
        annotate_empty(ax, "Buyer Cohort Retention Rate", "Không có dữ liệu retention.")

    # 2C. Inter-order gap
    ax = axes[2]
    if gaps is not None and len(gaps) > 0:
        med = float(np.nanmedian(gaps))
        ax.hist(gaps.clip(0, 500), bins=50, color=BLUE, edgecolor="white", alpha=0.8)
        ax.axvline(med, color="red", linestyle="--", linewidth=2, label=f"Median: {med:.0f} ngày")
        ax.set_xlabel("Số ngày giữa 2 lần mua (Gap days)")
        ax.set_ylabel("Tần suất")
        ax.set_title("Inter-Order Gap của Khách hàng Mua Lặp lại", fontweight="bold")
        ax.legend()
    else:
        annotate_empty(ax, "Inter-Order Gap", "Không có khách hàng mua lặp lại.")

    # 2D. Gross margin monthly
    ax = axes[3]
    if sales_month is not None and "gross_margin" in sales_month.columns and len(sales_month) > 0:
        gm = sales_month.copy()
        gm_pct = gm["gross_margin"].replace([np.inf, -np.inf], np.nan) * 100
        ax.plot(gm["date"], gm_pct, color=PURPLE, linewidth=2)
        ax.axhline(gm_pct.mean(), linestyle="--", color="gray", label=f"Trung bình: {gm_pct.mean():.1f}%")
        ax.axhline(0, color="black", linewidth=1)
        ax.fill_between(gm["date"], gm_pct, 0, where=(gm_pct < 0), alpha=0.25, color=RED)
        ax.set_ylabel("Gross Profit Margin (%)")
        ax.set_title("Biên Lợi nhuận Gộp theo Tháng", fontweight="bold")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    else:
        annotate_empty(ax, "Biên lợi nhuận gộp theo tháng", "Không tính được do thiếu Revenue/COGS.")

    # 2E. Hiệu quả khuyến mãi
    ax = axes[4]
    promo_cmp = pd.Series(dtype=float)
    if sales_promo is not None and "has_promo_day" in sales_promo.columns and "Revenue" in sales_promo.columns:
        promo_cmp = (
            sales_promo.groupby("has_promo_day")["Revenue"]
            .mean()
            .rename(index={False: "Không KM", True: "Có KM"})
        )

    if len(promo_cmp) > 0:
        labels = [idx for idx in ["Không KM", "Có KM"] if idx in promo_cmp.index]
        vals = promo_cmp.loc[labels].values
        colors = ["#B0BEC5" if x == "Không KM" else ORANGE for x in labels]
        bars = ax.bar(labels, vals / 1e6, color=colors, edgecolor="white", linewidth=1.2, alpha=0.9)
        ax.set_ylabel("Doanh thu TB/ngày (Triệu VND)")
        ax.set_title("Hiệu quả Khuyến mãi: Ngày có KM vs Không KM", fontweight="bold")
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{val/1e6:.2f}M",
                ha="center",
                fontsize=10,
                fontweight="bold",
            )
        if {"Có KM", "Không KM"}.issubset(set(promo_cmp.index)) and promo_cmp.loc["Không KM"] != 0:
            lift = promo_cmp.loc["Có KM"] / promo_cmp.loc["Không KM"] - 1
            ax.text(
                0.5,
                0.88,
                f"Promo lift = {lift:+.1%}",
                transform=ax.transAxes,
                ha="center",
                fontsize=11,
                bbox=dict(boxstyle="round", facecolor="white", edgecolor=BORDER, alpha=0.85),
            )
    else:
        annotate_empty(ax, "Hiệu quả khuyến mãi", "Không tính được promo day.")

    # 2F. Summary text
    ax = axes[5]
    ax.axis("off")

    median_gap = np.nanmedian(gaps) if gaps is not None and len(gaps) else np.nan
    ret_m1 = np.nan
    if retention is not None and not retention.empty and 1 in retention.columns:
        ret_m1 = retention[1].replace([np.inf, -np.inf], np.nan).mean()

    avg_gm = np.nan
    if sales_month is not None and "gross_margin" in sales_month.columns:
        avg_gm = sales_month["gross_margin"].replace([np.inf, -np.inf], np.nan).mean() * 100

    no_promo = promo_cmp.get("Không KM", np.nan) if len(promo_cmp) else np.nan
    yes_promo = promo_cmp.get("Có KM", np.nan) if len(promo_cmp) else np.nan
    lift = yes_promo / no_promo - 1 if pd.notna(no_promo) and no_promo != 0 and pd.notna(yes_promo) else np.nan

    def fmt_nan(value, fmt):
        return "N/A" if pd.isna(value) else fmt.format(value)

    summary_lines = [
        "TÓM TẮT NHANH",
        "",
        f"• M+1 buyer retention TB: {fmt_nan(ret_m1, '{:.1f}%')}",
        f"• Median inter-order gap: {fmt_nan(median_gap, '{:.0f} ngày')}",
        f"• Gross margin TB: {fmt_nan(avg_gm, '{:.1f}%')}",
        f"• Promo lift: {fmt_nan(lift, '{:+.1%}')}",
        "",
        "Lưu ý:",
        "• Ô trống ở heatmap là tháng tương lai/chưa đủ dữ liệu quan sát.",
        "• Buyer retention dùng mẫu số là khách đã mua ở M+0, nên M+0 thường bằng 100%.",
    ]
    ax.text(
        0.02,
        0.98,
        "\n".join(summary_lines),
        va="top",
        ha="left",
        fontsize=12,
        linespacing=1.5,
        bbox=dict(boxstyle="round,pad=0.8", facecolor="white", edgecolor=BORDER, alpha=0.95),
    )

    plt.suptitle("DASHBOARD 2 — XAI, Retention, Inter-order Gap, Margin & Promotion", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = Path(out_dir) / "dashboard_2_retention_promo_xai.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")
    return path


## 6. DASHBOARD 3

In [8]:
def save_dashboard3(base, xai=None, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rfm = base["rfm"]

    # Layout: hàng trên forecast 2023-2024 span toàn bộ, hàng dưới RFM + SHAP dependence + top features.
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

    # 3A. XAI-driven forecast to 2024 — đúng cửa sổ test của đề.
    ax1 = fig.add_subplot(gs[0, :])
    if xai is not None and xai.get("future_df") is not None and len(xai["future_df"]) > 0:
        hist = base["sales"][["Date", "Revenue"]].copy().sort_values("Date")
        hist_tail = hist[hist["Date"] >= (hist["Date"].max() - pd.Timedelta(days=540))]
        fut = xai["future_df"].copy().sort_values("Date")

        ax1.plot(
            hist_tail["Date"], hist_tail["Revenue"] / 1e6,
            color=BLUE, alpha=0.75, linewidth=1.1, label="Actual Revenue (train tail)"
        )
        ax1.plot(
            fut["Date"], fut["Revenue"] / 1e6,
            color=GREEN, alpha=0.95, linewidth=1.4, label="XAI Forecast Revenue"
        )
        if {"lower_12pct", "upper_12pct"}.issubset(fut.columns):
            ax1.fill_between(
                fut["Date"],
                fut["lower_12pct"] / 1e6,
                fut["upper_12pct"] / 1e6,
                color=GREEN, alpha=0.12,
                label=f"Risk band ±{xai.get('band_pct', 0.12):.0%}"
            )

        ax1.axvline(pd.Timestamp("2023-01-01"), color=RED, linestyle="--", linewidth=1.1, alpha=0.8)
        ax1.text(
            pd.Timestamp("2023-01-01"), ax1.get_ylim()[1] * 0.95,
            " Test window starts", color=RED, fontsize=9, va="top"
        )
        ax1.set_ylabel("Doanh thu (Triệu VND)")
        ax1.set_title(
            f"XAI-driven Forecast Optimization đến 2024  |  Valid MAE = {xai['metrics']['mae']:,.0f} VND, R² = {xai['metrics']['r2']:.3f}",
            fontweight="bold"
        )
        ax1.legend(fontsize=9, ncol=3)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax1.grid(True, alpha=0.3)
    elif xai is not None and "valid_df" in xai:
        v = xai["valid_df"].copy().tail(240)
        ax1.plot(v["Date"], v["Revenue"] / 1e6, color=BLUE, alpha=0.75, linewidth=1, label="Actual Revenue")
        ax1.plot(v["Date"], v["pred_optimized"] / 1e6, color=GREEN, alpha=0.85, linewidth=1.2, label="Validation Forecast")
        ax1.set_ylabel("Doanh thu (Triệu VND)")
        ax1.set_title(
            f"XAI-driven Forecast Optimization  |  MAE = {xai['metrics']['mae']:,.0f} VND",
            fontweight="bold"
        )
        ax1.legend(fontsize=9)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    else:
        annotate_empty(ax1, "XAI-driven Forecast Optimization đến 2024", "Chưa train XAI model hoặc thiếu forecast future_df.")

    # 3B. RFM distribution — pie giống fig09_rfm
    ax2 = fig.add_subplot(gs[1, 0])
    seg_count = rfm["rfm_segment"].value_counts()
    if len(seg_count) > 0:
        ax2.pie(
            seg_count.values, labels=seg_count.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2", len(seg_count)),
            startangle=90, pctdistance=0.75,
            textprops={"fontsize": 8}
        )
        ax2.set_title("Phân bổ Khách hàng theo Phân khúc RFM", fontweight="bold")
    else:
        annotate_empty(ax2, "Phân bổ RFM", "Không có dữ liệu RFM.")

    # 3C. max_discount_lag1 SHAP value × n_orders_lag1 — dependence style, forecast-safe.
    ax3 = fig.add_subplot(gs[1, 1])
    dep_feat = "max_discount_lag1" if xai is not None and "max_discount_lag1" in xai.get("feature_cols", []) else "max_discount_rmean7"
    if xai is not None and dep_feat in xai["X_samp"].columns and dep_feat in xai["feature_cols"]:
        feat_idx = xai["feature_cols"].index(dep_feat)
        xs = xai["X_samp"][dep_feat].values
        ys = xai["shap_values"][:, feat_idx]
        color_feat = "n_orders_lag1" if "n_orders_lag1" in xai["X_samp"].columns else None
        if color_feat:
            c = xai["X_samp"][color_feat].values
            sc = ax3.scatter(xs, ys, c=c, cmap="coolwarm", alpha=0.4, s=8, edgecolors="none")
            cb = fig.colorbar(sc, ax=ax3, fraction=0.046, pad=0.02)
            cb.set_label(color_feat)
        else:
            ax3.scatter(xs, ys, color=BLUE, alpha=0.4, s=8, edgecolors="none")
        ax3.axhline(0, color="gray", linewidth=1, linestyle="--")
        ax3.set_title(f"{dep_feat} SHAP value × n_orders_lag1", fontweight="bold")
        ax3.set_xlabel(dep_feat)
        ax3.set_ylabel("SHAP value")
    else:
        annotate_empty(ax3, "max_discount_lag1 SHAP value × n_orders_lag1", "Không có SHAP hoặc thiếu feature forecast-safe.")

    # 3D. Top 10 SHAP features — barh giống summary dashboard
    ax4 = fig.add_subplot(gs[1, 2])
    if xai is not None and "shap_df" in xai and len(xai["shap_df"]) > 0:
        t10 = xai["shap_df"].head(10).sort_values("mean_abs_shap", ascending=True)
        ax4.barh(t10["feature"], t10["mean_abs_shap"], color=BLUE, alpha=0.85)
        ax4.set_title("Top 10 SHAP Features", fontweight="bold")
        ax4.set_xlabel("Mean |SHAP|")
        ax4.tick_params(axis="y", labelsize=8)
        for i, v in enumerate(t10["mean_abs_shap"].values):
            ax4.text(v, i, f" {v:.3f}", va="center", fontsize=8)
    else:
        annotate_empty(ax4, "Top 10 SHAP Features", "Không có XAI model.")

    plt.suptitle("DASHBOARD 3 — RFM Strategy & XAI-driven Forecast to 2024", fontsize=15, fontweight="bold", y=0.98)
    path = Path(out_dir) / "dashboard_3_rfm_xai_forecast_2024.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")
    return path


## 7. RUN ALL

In [ ]:
def run_all(data_dir=DATA_DIR, out_dir=OUT_DIR, train_xai=True):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    data = load_data(data_dir)
    base = prepare_base(data)
    xai = train_xai_model(base) if train_xai else None

    paths = [
        save_dashboard1(base, out_dir),
        save_dashboard2(base, xai, out_dir),
        save_dashboard3(base, xai, out_dir),
    ]

    print("\nDONE. Ảnh dashboard đã xuất:")
    for p in paths:
        print(" -", Path(p).resolve())


    return base, xai, paths


## Chạy toàn bộ và xuất ảnh

In [10]:
# Nếu CSV nằm ở thư mục khác, sửa tại đây, ví dụ:
# DATA_DIR = r"D:/datathon/raw"

base, xai, dashboard_paths = run_all(DATA_DIR, OUT_DIR, train_xai=True)
dashboard_paths


Using DATA_DIR = C:\Users\admin\Documents\python\mycode\raw
Saved: dashboard_outputs\dashboard_1_revenue_inventory_category.png
Saved: dashboard_outputs\dashboard_2_retention_promo_xai.png
Saved: dashboard_outputs\dashboard_3_rfm_xai_forecast_2024.png
Saved submission: dashboard_outputs\submission_xai_2024.csv

DONE. Ảnh dashboard đã xuất:
 - C:\Users\admin\Documents\python\notebook\dashboard_outputs\dashboard_1_revenue_inventory_category.png
 - C:\Users\admin\Documents\python\notebook\dashboard_outputs\dashboard_2_retention_promo_xai.png
 - C:\Users\admin\Documents\python\notebook\dashboard_outputs\dashboard_3_rfm_xai_forecast_2024.png
Submission: C:\Users\admin\Documents\python\notebook\dashboard_outputs\submission_xai_2024.csv


[WindowsPath('dashboard_outputs/dashboard_1_revenue_inventory_category.png'),
 WindowsPath('dashboard_outputs/dashboard_2_retention_promo_xai.png'),
 WindowsPath('dashboard_outputs/dashboard_3_rfm_xai_forecast_2024.png')]